# 8. 신경망 주택 가격 회귀

표준화와 MLPRegressor로 주택 가격을 예측합니다. 은닉층 크기와 실제 가중치 행렬을 살펴보고 소규모 데이터의 평가 한계를 확인합니다.

원본 노트북 Lv2_Python_1119_Regression_nn_house_price.ipynb

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

from sklearn.neural_network import MLPRegressor

In [ ]:
from pathlib import Path
import pandas as pd

data_dir = DATA_DIR if 'DATA_DIR' in globals() else Path('data')
if not (data_dir / 'house.price.csv').is_file():
    data_dir = Path('Study_Python') / '[7] Machine_Learning' / 'data'
if not (data_dir / 'house.price.csv').is_file():
    data_dir = Path('..') / 'Study_Python' / '[7] Machine_Learning' / 'data'
file_path = data_dir / 'house.price.csv'
df = pd.read_csv(file_path)

In [ ]:
# 불필요한 열(ID) 제거
# df = df.drop(columns=["ID"])

In [ ]:
# 독립 변수(X) 종속 변수(y) 설정
X = df[["year", "size"]].values  # 입력 변수: year, size
y = df["price"].values  # 예측 대상: price

In [ ]:
# 데이터 분할 (훈련: 80%, test: 20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
y_train

In [ ]:
X_train

In [ ]:
# 스케일링 (표준화)
scaler_X = StandardScaler()

X_train_scaled = scaler_X.fit_transform(X_train) # 훈련 데이터에서 평균, 표준편차를 구하고 변환
X_test_scaled = scaler_X.transform(X_test) # 훈련 데이터에서 구한 기준으로 테스트 데이터 변환 <-- 데이터 누수 방지

In [ ]:
X_train_scaled

In [ ]:
# 신경망 모델 - MLPRegressor 사용 (Multi-Layer Perceptron Regressor)
# Perceptron - 다수의 입력을 받아 하나의 이진 출력을 내는 선형 분류 알고리즘
# 회귀용 신경망을 간편하게 구성, 학습
# cf, MLPClassifier

mlp = MLPRegressor(
    hidden_layer_sizes=(5,),  # note 5개 은닉층 1
    activation="relu",        # ReLU 활성화 함수 -> 비선형 함수 ( _/ 형태의 함수 0이하는 다 0, 양수는 양수 값을 출력)
    solver="adam",
    max_iter=500,             # 500회 에포크 학습 -> 무조건 500번 반복이 아님. 500번 이전에 최적의 결과가 도출되면 조기 종료
    random_state=42
)

# 입력층 노드 수?
# -> 2 개의 파라미터 (["year", "size"])
# 출력층 노드 수?
# -> 1 개 ["price"]

# 분석가 지정하 하이퍼파라미터 : 노드 수, 활성화 함수, 최대 학습 횟수
# 학습을 통해 결정되는 파라미터	: 가중치, 편향

In [ ]:
# 모델 설정 확인

print(mlp.get_params())
# print(mlp.hidden_layer_sizes)
# print(mlp.activation)

In [ ]:
print(mlp.hidden_layer_sizes)

In [ ]:
# 모델 학습
mlp.fit(X_train_scaled, y_train)

In [ ]:
# 예측 수행 (스케일링된 값)
y_pred = mlp.predict(X_test_scaled)

In [ ]:
# 성능 지표 계산
mse = mean_squared_error(y_test, y_pred)
rmse = mse ** 0.5  # RMSE는 MSE의 제곱근
mae = mean_absolute_error(y_test, y_pred)

# 결과 출력
print(f"MSE: {mse:.2f}, RMSE: {rmse:.2f}, MAE: {mae:.2f}")


# 이 신경망이 학습하는 파라미터는 가중치 15개 + 편향 6개 = 총 21개

In [ ]:
# 학습된 가중치와 편향 출력
print("\n===== 가중치(Weights) =====")
for i, weight_matrix in enumerate(mlp.coefs_):
    print(f"Layer {i} -> {i+1} 가중치 shape {weight_matrix.shape}:\n", weight_matrix)

print("\n===== 편향(Biases) =====")
for i, bias_vector in enumerate(mlp.intercepts_):
    print(f"Layer {i+1} 편향 shape {bias_vector.shape}:\n", bias_vector)

In [ ]:
###### 데이터 증강(Augmentation)
##### 10개의 데이터로 학습???

데이터 증강은 훈련 세트에만 적용합니다. 테스트 데이터는 분할 시점의 관측값을 그대로 사용합니다.

In [ ]:
import numpy as np
import pandas as pd

# 랜덤 노이즈 추가 (±5% 범위 내)
np.random.seed(42)
noise_factor = 0.05  # 5% 노이즈 추가

train_data = pd.DataFrame(X_train, columns=['year', 'size'])
train_data['price'] = y_train
df_augmented = train_data.copy()
df_augmented["year"] += df_augmented["year"] * np.random.uniform(-noise_factor, noise_factor, size=len(train_data))
df_augmented["price"] += df_augmented["price"] * np.random.uniform(-noise_factor, noise_factor, size=len(train_data))
df_augmented["size"] += df_augmented["size"] * np.random.uniform(-noise_factor, noise_factor, size=len(train_data))

# 원본 데이터와 합쳐서 새로운 데이터셋 생성
df_final = pd.concat([train_data, df_augmented], ignore_index=True)

# 증폭된 데이터 개수 확인
print(f"증폭된 데이터 개수: {len(df_final)}")
df_final.head()
    # 너무 과도한 변형및 증강은 역효과... 적절한 수준으로 증강해야 (2~3배?)